# GenomicsCopilot — variant interpretation walkthrough

This notebook demonstrates the `variant_interpreter` agent end-to-end using the bundled sample VCF.

> ⚠️ Research / educational use only. Not for clinical use. See `DISCLAIMER.md`.

## Setup

Make sure you have installed the package with the demo extras and set `ANTHROPIC_API_KEY` in `.env`:

```bash
pip install -e ".[demo]"
cp .env.example .env   # then edit .env
```

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from agentic_genomics.agents.variant_interpreter import run_variant_interpreter

## Run the agent on the demo VCF

We supply two HPO terms for the hypothetical proband:
- `HP:0001250` — Seizure
- `HP:0001263` — Global developmental delay

In [ ]:
state = run_variant_interpreter(
    vcf_path="../data/samples/proband_demo.vcf",
    hpo_terms=["HP:0001250", "HP:0001263"],
    max_variants=25,
)
state.variants[:3]

## Inspect the reasoning trace

Every node in the graph appends a structured entry here. This is the audit trail you'd include in a downstream report or MLflow run.

In [ ]:
import pandas as pd

pd.DataFrame(state.reasoning_trace)[["node", "summary"]]

## Inspect per-variant evidence

Each `AnnotatedVariant` carries population, functional, clinical, phenotype, and ACMG evidence.

In [ ]:
rows = []
for av in state.variants:
    rows.append({
        "variant": av.variant.key,
        "gene": av.variant.gene,
        "gnomad_af": av.population.gnomad_af,
        "cadd": av.functional.cadd_phred,
        "revel": av.functional.revel,
        "clinvar": av.clinical.clinvar_significance,
        "phenotype_match": av.phenotype.match_strength,
        "acmg_call": av.acmg.call if av.acmg else None,
        "acmg_criteria": ", ".join(av.acmg.criteria_triggered) if av.acmg else "",
    })
pd.DataFrame(rows)

## The LLM-synthesised report

This is the artefact a human analyst would read. Note the explicit evidence chains and calibrated language.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(state.report_markdown))

## Next steps

- Swap in your own VCF (public/synthetic only — see `DISCLAIMER.md`).
- Tune phenotype terms to see how ranking changes.
- Inspect `state.reasoning_trace[*]["evidence"]` for the full machine-readable audit trail.
- Read `docs/why-agentic.md` for the design philosophy.